In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv(r"C:\Users\chara\desktop\Machine_learning\Logistic_Regression\loan_data.csv")

In [3]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [4]:
x=df.drop(columns='loan_status')
y=df.loan_status

In [5]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,random_state=42,train_size=0.2)

In [6]:
obj_col=x.select_dtypes(include='string').columns

In [7]:
x[obj_col].nunique()

Series([], dtype: float64)

In [8]:
x['person_education'].unique()

array(['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'],
      dtype=object)

In [9]:
order= ['High School','Associate','Bachelor','Master','Doctorate']

In [10]:
preprocessing=ColumnTransformer(
    transformers=[
        ('OneHotEncoder',OneHotEncoder(handle_unknown="ignore"),
                ['person_gender','person_home_ownership','loan_intent','previous_loan_defaults_on_file']),
        ('ordinal',OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1),
                ['person_education'])
    ],remainder="passthrough"
)

In [11]:
pipeline = Pipeline(
    steps=[("preprocessing",preprocessing),
          ('model',RandomForestClassifier(random_state=42))
          ]
        )

In [18]:
grid_search_cv=GridSearchCV(
    estimator=pipeline,
    param_grid={
        'model__n_estimators':[100,200],
        'model__max_depth':[None,10,20],
        'model__min_samples_split':[2,5],
        'model__min_samples_leaf':[1,2],
        'model__max_features':['sqrt'],
        'model__class_weight':[None,'balanced']
    },
    verbose=10,
    n_jobs=-1 
)

In [19]:
grid_search_cv.fit(xtrain,ytrain)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__class_weight': [None, 'balanced'], 'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt'], 'model__min_samples_leaf': [1, 2], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation

In [20]:
grid_search_cv.best_estimator_ # It will return the best model/estimator

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('OneHotEncoder', ...), ('ordinal', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the dif

In [21]:
grid_search_cv.best_params_ # it will return the best parameter values

{'model__class_weight': None,
 'model__max_depth': None,
 'model__max_features': 'sqrt',
 'model__min_samples_leaf': 2,
 'model__min_samples_split': 5,
 'model__n_estimators': 200}

In [22]:
grid_search_cv.cv_results_

{'mean_fit_time': array([1.00435519, 1.93013444, 0.9446444 , 1.89197702, 0.9378171 ,
        1.82833524, 0.93767905, 1.82678308, 0.82203937, 1.68992929,
        0.81388607, 1.62927265, 0.81278162, 1.66344399, 0.79974914,
        1.58554821, 0.99275351, 1.96464615, 0.99104252, 1.88725839,
        0.96988268, 1.85592456, 0.96703668, 1.97072811, 1.0547585 ,
        2.0796494 , 1.00865397, 1.99611382, 1.02479053, 2.102109  ,
        0.98761072, 1.99254174, 0.90154142, 1.78021946, 0.90697703,
        1.66817122, 0.90299306, 1.78271794, 0.86954384, 1.93987131,
        1.18810358, 2.32193918, 1.08438907, 2.14451022, 1.06803436,
        2.28835144, 1.25794945, 1.58876133]),
 'std_fit_time': array([0.03312545, 0.03431983, 0.03441712, 0.04414397, 0.01105089,
        0.02716862, 0.02099577, 0.06396214, 0.03898193, 0.09024607,
        0.02462857, 0.0871224 , 0.04214647, 0.0676712 , 0.0248038 ,
        0.0735881 , 0.05193661, 0.09128992, 0.02768517, 0.09031896,
        0.04364   , 0.12512945, 0.058

In [23]:
results = pd.DataFrame(grid_search_cv.cv_results_) 

In [ ]:
# results

In [24]:
results.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__class_weight,param_model__max_depth,param_model__max_features,param_model__min_samples_leaf,param_model__min_samples_split,param_model__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
7,1.826783,0.063962,0.079014,0.006918,None,None,sqrt,2,5,200,"{'model__class_weight': None, 'model__max_dept...",0.920556,0.923889,0.923333,0.925000,0.922778,0.923111,0.001474,1
23,1.970728,0.074133,0.083055,0.008874,None,20,sqrt,2,5,200,"{'model__class_weight': None, 'model__max_dept...",0.918889,0.921111,0.929444,0.926667,0.919444,0.923111,0.004196,1
18,0.991043,0.027685,0.051804,0.005739,None,20,sqrt,1,5,100,"{'model__class_weight': None, 'model__max_dept...",0.917778,0.923333,0.925556,0.928333,0.920000,0.923000,0.003778,3
19,1.887258,0.090319,0.080602,0.003870,None,20,sqrt,1,5,200,"{'model__class_weight': None, 'model__max_dept...",0.917778,0.922778,0.929444,0.925000,0.918333,0.922667,0.004338,4
3,1.891977,0.044144,0.083041,0.005063,None,None,sqrt,1,5,200,"{'model__class_weight': None, 'model__max_dept...",0.917222,0.921111,0.927222,0.927778,0.919444,0.922556,0.004225,5
1,1.930134,0.034320,0.081051,0.003333,None,None,sqrt,1,2,200,"{'model__class_weight': None, 'model__max_dept...",0.920000,0.919444,0.927222,0.927222,0.917778,0.922333,0.004058,6
5,1.828335,0.027169,0.079838,0.006043,None,None,sqrt,2,2,200,"{'model__class_weight': None, 'model__max_dept...",0.917778,0.922222,0.925000,0.925000,0.920000,0.922000,0.002824,7
13,1.663444,0.067671,0.074753,0.008838,None,10,sqrt,2,2,200,"{'model__class_weight': None, 'model__max_dept...",0.918333,0.919444,0.923333,0.927778,0.920000,0.921778,0.003432,8
41,2.321939,0.165586,0.099454,0.008865,balanced,20,sqrt,1,2,200,"{'model__class_weight': 'balanced', 'model__ma...",0.917222,0.923333,0.923889,0.926667,0.917778,0.921778,0.003675,8
0,1.004355,0.033125,0.052960,0.008892,None,None,sqrt,1,2,100,"{'model__class_weight': None, 'model__max_dept...",0.918333,0.920556,0.924444,0.927222,0.917778,0.921667,0.003635,10


In [25]:
grid_search_cv.predict(xtrain)

array([1, 0, 0, ..., 0, 1, 0], shape=(9000,))